In [23]:
import pandas as pd

In [24]:
deliveries = pd.read_csv("../datasets/deliveries_cleaned.csv")
matches = pd.read_csv("../datasets/matches_cleaned.csv")

In [25]:
#aggregate batting stats per match
batting_match = (
    deliveries
    .groupby(['match_id', 'batter'])
    .agg(
        runs=('batsman_runs', 'sum'),
        balls=('legal_ball', 'sum'),
        fours=('batsman_runs', lambda x: (x == 4).sum()),
        sixes=('batsman_runs', lambda x: (x == 6).sum()),
        outs=('is_wicket', 'sum')
    )
    .reset_index()
)


In [26]:
#add strike rate
batting_match['strike_rate'] = (
    batting_match['runs'] / batting_match['balls'].replace(0, 1) * 100
)


In [27]:
#Rolling last 5 matches form
batting_match = batting_match.sort_values(['batter', 'match_id'])

batting_match['runs_last5'] = (
    batting_match
    .groupby('batter')['runs']
    .rolling(5, min_periods=1)
    .mean()
    .reset_index(drop=True)
)

batting_match['sr_last5'] = (
    batting_match
    .groupby('batter')['strike_rate']
    .rolling(5, min_periods=1)
    .mean()
    .reset_index(drop=True)
)


In [28]:
#venue averages
#merge venue info
batting_match = batting_match.merge(
    matches[['id', 'venue']],
    left_on='match_id',
    right_on='id',
    how='left'
)



In [29]:
#venue wise player average
venue_avg = (
    batting_match
    .groupby(['batter', 'venue'])['runs']
    .mean()
    .reset_index()
    .rename(columns={'runs': 'venue_avg_runs'})
)

batting_match = batting_match.merge(
    venue_avg,
    on=['batter', 'venue'],
    how='left'
)


In [30]:
#opponent specific stats
#add opponent
deliveries = deliveries.merge(
    matches[['id', 'team1', 'team2']],
    left_on='match_id',
    right_on='id',
    how='left'
)

deliveries['opponent'] = deliveries.apply(
    lambda x: x['team2'] if x['batting_team'] == x['team1'] else x['team1'],
    axis=1
)


In [31]:
#opponent wise runs
pvt = (
    deliveries
    .groupby(['match_id', 'batter', 'opponent'])['batsman_runs']
    .sum()
    .reset_index()
)

pvt_avg = (
    pvt
    .groupby(['batter', 'opponent'])['batsman_runs']
    .mean()
    .reset_index()
    .rename(columns={'batsman_runs': 'pvt_avg_runs'})
)

batting_match = batting_match.merge(
    pvt_avg,
    on='batter',
    how='left'
)


In [32]:
#career stats
career_stats = (
    batting_match
    .groupby('batter')
    .agg(
        career_runs=('runs', 'sum'),
        career_matches=('match_id', 'nunique'),
        career_avg=('runs', 'mean'),
        career_sr=('strike_rate', 'mean')
    )
    .reset_index()
)

batting_match = batting_match.merge(
    career_stats,
    on='batter',
    how='left'
)


In [33]:
#create training labels(next match runs)
batting_match = batting_match.sort_values(['batter', 'match_id'])

batting_match['next_match_runs'] = (
    batting_match
    .groupby('batter')['runs']
    .shift(-1)
)


In [34]:
#remove last match per player
batting_match = batting_match.dropna(subset=['next_match_runs'])


In [35]:
#Train-Test Split (Time-Series Aware)
cutoff_match = batting_match['match_id'].quantile(0.8)

train = batting_match[batting_match['match_id'] <= cutoff_match]
test  = batting_match[batting_match['match_id'] > cutoff_match]


In [36]:
#feature pipeline (preprocessing)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
import joblib


In [37]:
#numeric features
num_features = [
    'runs_last5', 'sr_last5', 'venue_avg_runs',
    'career_avg', 'career_sr'
]

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features)
])


In [38]:
#save pipeline
joblib.dump(preprocessor, "../feature_pipeline.pkl")


['../feature_pipeline.pkl']

In [39]:
#final feature engineered dataset
final_dataset = batting_match[
    num_features + ['next_match_runs']
]

final_dataset.to_csv("../datasets/dataset.csv", index=False)


In [40]:
matches.columns

Index(['id', 'season', 'city', 'date', 'match_type', 'player_of_match',
       'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner',
       'result', 'result_margin', 'target_runs', 'target_overs', 'super_over',
       'method', 'umpire1', 'umpire2', 'year'],
      dtype='object')